![imagenes](logo.png)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# --- Cargar CSV (ajusta la ruta si hace falta) ---
df = pd.read_csv("datos_preproc_demo.csv")
df.columns


In [ ]:
df.head()

In [ ]:
# En esta etapa NO usamos y/objetivo; es solo preprocesamiento
X_train, X_test = train_test_split(df, test_size=0.25, random_state=0)

In [ ]:
# Radiografía de faltantes (conteo y porcentaje)

faltantes = X_train.isna().sum().sort_values(ascending=False)
porcentaje = (X_train.isna().mean()*100).round(2).sort_values(ascending=False)
resumen_faltantes = pd.DataFrame({"faltantes": faltantes, "porcentaje_%": porcentaje})
print("\nResumen de faltantes por columna:\n")
print(resumen_faltantes, "\n")

In [ ]:
# Columnas con atípicos

num_cols = X_train.select_dtypes(include=[np.number]).columns

cols_con_outliers = []
for c in num_cols:
    s = pd.to_numeric(X_train[c], errors="coerce").dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = int(((s < lim_inf) | (s > lim_sup)).sum())
    if n_out > 0:
        cols_con_outliers.append(c)

cols_con_outliers

In [ ]:
df.columns

In [ ]:
# --- Listas de columnas ---

# Numéricas
# num_med_rob_cols: inputacion mediana con escalado robusto
# num_mean_min_cols: inputacion media con escalado min_max
# num_mean_std_cols: inputacion media con escalado std

# Categóricas
# cat_ohe_cols: inputacion moda con OneHot
# cat_ord_cols: inputacion moda con Ordinal

num_med_rob_cols = []   # mediana + Robust
num_mean_min_cols  = []    # media + MinMax
num_mean_std_cols  = []    # media + Standard

cat_ohe_cols = []                  # moda + OneHot
cat_ord_cols = []                  # moda + Ordinal

#########################################################
#########################################################
#########################################################

passthrough_cols = []                      # pasar sin procesar
drop_cols        = []                         # eliminar


categorias_ordinales = [ ]

In [ ]:
# --- Pipelines NUMÉRICOS ---
pipe_med_rob = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler())
])

pipe_mean_min = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  MinMaxScaler())
])

pipe_mean_std = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler())
])

# --- Pipelines CATEGÓRICOS ---
pipe_cat_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # moda
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

pipe_cat_ord = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # moda
    ("encoder", OrdinalEncoder(categories=categorias_ordinales,
                               handle_unknown="use_encoded_value", unknown_value=-1))
])


In [ ]:
# --- ColumnTransformer unificado ---
preprocessor = ColumnTransformer(
    transformers=[
        # Numéricos (6 columnas -> 3 bloques con su imputer+scaler)
        ("num_med_rob", pipe_med_rob, num_med_rob_cols),
        ("num_mean_min",  pipe_mean_min,  num_mean_min_cols),
        ("num_mean_std",  pipe_mean_std,  num_mean_std_cols),

        # Categóricos (4 columnas -> 2 OneHot y 2 Ordinal, todos con moda)
        ("cat_ohe",      pipe_cat_ohe,  cat_ohe_cols),
        ("cat_ord",      pipe_cat_ord,  cat_ord_cols),

        # Passthrough (sin preprocesar)
        ("passthrough",  "passthrough", passthrough_cols),

        # Drop explícito de la columna con ~90% NA
        ("drop_high_na", "drop",        drop_cols),
    ],
    remainder="drop",                        # descarta cualquier otra columna no listada
    verbose_feature_names_out=False
)

In [ ]:
# ---------- Ajuste y transformación ----------
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print("Shape train ->", X_train_proc.shape)
print("Shape test  ->", X_test_proc.shape)


In [ ]:
# Reconstruir DataFrame con nombres de columnas
cols_out = preprocessor.get_feature_names_out()
X_train_proc_df = pd.DataFrame(X_train_proc, columns=cols_out)
X_test_proc_df = pd.DataFrame(X_test_proc, columns=cols_out)

In [ ]:
X_test_proc_df

In [ ]:
# Verificar que no quedan NA tras el preprocesamiento 
print(pd.isna(X_train_proc).sum(), "NA en matriz transformada (debería ser 0)")